In [1]:
import polars as pl

import nwec.utility_reporting.arrearage_counts
import nwec.utility_reporting.arrearages
import nwec.utils.excel
from nwec.constants import RAW_UTILITY_DATA, Utility

YEAR = 2024
QUARTER = 4
NUM_MONTHS = 12
COLS_PER_MONTH = 1
SHEET_SEARCH_STRING = "disconnections"
NUM_DISCONNECTIONS_SEARCH_STRING = "number of disconnections"
NUM_NOTICES_SEARCH_STRING = "number of customers by customer class receiving disconnection notices"
spreadsheet = RAW_UTILITY_DATA / str(YEAR) / f"{Utility.CNG.code}_{YEAR}_Q{QUARTER}.xlsx"
source_date_format = "%Y-%m-%d %H:%M:%S"

In [2]:
sheet_index = nwec.utils.excel.get_sheet_index_from_name(spreadsheet, SHEET_SEARCH_STRING)
df = pl.read_excel(spreadsheet, sheet_id=sheet_index, has_header=False)
_, start_index = nwec.utils.excel.find_cell_by_string(df, NUM_DISCONNECTIONS_SEARCH_STRING)
disconnections = df.select(df.columns[start_index : start_index + NUM_MONTHS * COLS_PER_MONTH])

# Number of Disconnections


In [3]:
date_row = nwec.utility_reporting.arrearages.infer_date_row(disconnections, source_date_format)
disconnections = disconnections.tail(-date_row)  # remove rows before the date row
disconnections = nwec.utility_reporting.arrearage_counts.format_arrearage_count_dates(
    disconnections, source_date_format
)
disconnections = nwec.utility_reporting.arrearages.add_zip_and_customer_class_cols(df, disconnections)
disconnections = nwec.utility_reporting.arrearage_counts.normalize_arrearage_count_cols(disconnections, Utility.CNG)

[0, 15, 23]


/home/peter/coding/nwec/nwec/utility_reporting/arrearages/arrearages.py:128: UserWarning: Multiple columns have at least 5 rows that match the ZIP code pattern; using the first.
  zip_column = spreadsheet_df.select(pl.nth(nwec.utils.excel.infer_zip_column(spreadsheet_df, start_col=start_col)))
/home/peter/coding/nwec/nwec/utility_reporting/arrearages/arrearages.py:133: UserWarning: Multiple columns have at least 5 rows that match the customer class pattern; using the first.
  pl.nth(infer_customer_class_column(spreadsheet_df, start_col=start_col))


In [4]:
disconnections

Zip Code,Year,Month,Count,Utility,Customer Class
str,i32,i32,i32,str,str
"""98220""",2024,1,0,"""Cascade Natural Gas Corporatio…","""Residential"""
"""98221""",2024,1,7,"""Cascade Natural Gas Corporatio…","""Residential"""
"""98223""",2024,1,3,"""Cascade Natural Gas Corporatio…","""Residential"""
"""98225""",2024,1,14,"""Cascade Natural Gas Corporatio…","""Residential"""
"""98226""",2024,1,7,"""Cascade Natural Gas Corporatio…","""Residential"""
…,…,…,…,…,…
"""99350""",2024,12,0,"""Cascade Natural Gas Corporatio…","""Residential"""
"""99352""",2024,12,0,"""Cascade Natural Gas Corporatio…","""Residential"""
"""99353""",2024,12,0,"""Cascade Natural Gas Corporatio…","""Residential"""


# Number of Disconnection Notices


In [5]:
_, start_index = nwec.utils.excel.find_cell_by_string(df, NUM_NOTICES_SEARCH_STRING)
notices = df.select(df.columns[start_index + 2 : start_index + NUM_MONTHS * COLS_PER_MONTH + 2])

In [7]:
date_row = nwec.utility_reporting.arrearages.infer_date_row(notices, source_date_format)
notices = notices.tail(-date_row)  # remove rows before the date row
notices = nwec.utility_reporting.arrearage_counts.format_arrearage_count_dates(notices, source_date_format)
notices = nwec.utility_reporting.arrearages.add_zip_and_customer_class_cols(
    df, notices, start_col=NUM_MONTHS * COLS_PER_MONTH + 7
)
notices = nwec.utility_reporting.arrearage_counts.normalize_arrearage_count_cols(notices, Utility.CNG)

[23]


# Save Results


In [9]:
# nwec.utility_reporting.arrearage_counts.save_processed_arrearage_counts(arrearage_counts)